# Real-Time Nut Detection, Separation, Tracking & Counting

**Computer Vision Project using OpenCV**

## Project Goal
Develop a vision system that detects metal nuts moving on a blue conveyor, separates touching nuts, tracks them, and increases a cumulative counter by **1** every time a nut crosses the counting line.

### Main Challenges
- Nuts may touch each other.
- Some nuts may stand on their side.
- The operator's hand appears in the lower part of the video.
- The conveyor motion is **upward** in the image.
- The same nut must be counted only once.

## Stage 0 — Imports & Project Setup

This notebook uses:
- **OpenCV** for image/video processing.
- **NumPy** for arrays and kernels.
- **scikit-image** for local peaks and watershed.
- **SciPy** for marker labeling and Hungarian assignment.

> Put `nuts_video.mp4` in the same folder as this notebook before running.

In [ ]:
import cv2
import numpy as np
import os

from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

print("OpenCV Version:", cv2.__version__)
print("Video exists:", os.path.exists("nuts_video.mp4"))

## Stage 1 — Video Acquisition

A video is processed **frame by frame**.  
Each frame is treated as an image and passed through the computer vision pipeline.

In [ ]:
VIDEO_PATH = "nuts_video.mp4"

cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = total_frames / fps if fps else 0

print("FPS:", fps)
print("Width:", width)
print("Height:", height)
print("Total Frames:", total_frames)
print("Duration:", round(duration, 2), "seconds")

cap.release()

## Stage 2 — Read and Display a Sample Frame

OpenCV reads images in **BGR**, while Matplotlib displays **RGB**.  
Therefore, the frame is converted before visualization.

In [ ]:
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(VIDEO_PATH)
ret, sample_frame = cap.read()
cap.release()

if not ret:
    raise RuntimeError("Could not read sample frame.")

sample_rgb = cv2.cvtColor(sample_frame, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(5, 8))
plt.imshow(sample_rgb)
plt.title("Original Sample Frame")
plt.axis("off")
plt.show()

## Stage 3 — HSV Analysis

The blue conveyor has high saturation, while the metallic nuts have lower saturation.  
Therefore, the **Saturation (S) channel** is useful for separating nuts from the conveyor.

In [ ]:
hsv = cv2.cvtColor(sample_frame, cv2.COLOR_BGR2HSV)
H, S, V = cv2.split(hsv)

plt.figure(figsize=(12, 4))

for i, (img, title) in enumerate([(H, "Hue"), (S, "Saturation"), (V, "Value")], start=1):
    plt.subplot(1, 3, i)
    plt.imshow(img, cmap="gray")
    plt.title(title)
    plt.axis("off")

plt.tight_layout()
plt.show()

## Stage 4 — Binary Segmentation

Pixels with low saturation are classified as possible metallic nuts.

- Foreground (nut) → **White**
- Conveyor background → **Black**

In [ ]:
SATURATION_THRESHOLD = 100

mask = cv2.inRange(S, 0, SATURATION_THRESHOLD)

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(sample_rgb)
plt.title("Original")

plt.subplot(1, 2, 2)
plt.imshow(mask, cmap="gray")
plt.title("Binary Mask")

for i in [1, 2]:
    plt.subplot(1, 2, i)
    plt.axis("off")

plt.tight_layout()
plt.show()

## Stage 5 — Region of Interest (ROI)

The operator's hand may enter the lower part of the frame.  
To reduce false detections, the system processes only the upper region.

The final counting line is placed at:

**`LINE_Y = 300`**

The conveyor motion is **upward**, so a nut is counted when its center moves from:

`old_y > LINE_Y` → `new_y <= LINE_Y`

In [ ]:
LINE_Y = 300
PROCESSING_BOTTOM = 370

roi = sample_frame[:PROCESSING_BOTTOM, :]
roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(5, 7))
plt.imshow(roi_rgb)
plt.axhline(LINE_Y, linewidth=2)
plt.title("Processing ROI and Counting Line")
plt.axis("off")
plt.show()

## Stage 6 — Morphological Opening

After segmentation, small white artifacts may remain.

**Opening = Erosion → Dilation**

A `3×3` kernel removes small isolated noise while preserving the nut shapes.

In [ ]:
kernel = np.ones((3, 3), np.uint8)

roi_hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
_, roi_s, _ = cv2.split(roi_hsv)

roi_mask = cv2.inRange(roi_s, 0, SATURATION_THRESHOLD)

opened_mask = cv2.morphologyEx(
    roi_mask,
    cv2.MORPH_OPEN,
    kernel
)

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(roi_mask, cmap="gray")
plt.title("Before Opening")

plt.subplot(1, 2, 2)
plt.imshow(opened_mask, cmap="gray")
plt.title("After Opening")

for i in [1, 2]:
    plt.subplot(1, 2, i)
    plt.axis("off")

plt.tight_layout()
plt.show()

## Stage 7 — Contours & Feature Extraction

Contours represent connected foreground objects.

For each contour we can calculate:
- Area
- Width and height
- Bounding box
- Center / centroid

This stage revealed an important problem: **touching nuts can become one merged contour**.

In [ ]:
contours, _ = cv2.findContours(
    opened_mask,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

print("Detected contours:", len(contours))

for i, c in enumerate(contours[:10], start=1):
    area = cv2.contourArea(c)
    x, y, w, h = cv2.boundingRect(c)
    print(f"Object {i}: Area={area:.1f}, Size={w}x{h}")

## Stage 8 — Fill Nut Holes

The physical hole of the nut is useful visually, but it makes the distance transform produce a ring.

For separation only, a **filled version of the mask** is created so each nut behaves like a solid object.

In [ ]:
filled_mask = opened_mask.copy()

fill_contours, _ = cv2.findContours(
    filled_mask,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

cv2.drawContours(
    filled_mask,
    fill_contours,
    -1,
    255,
    cv2.FILLED
)

plt.figure(figsize=(5, 7))
plt.imshow(filled_mask, cmap="gray")
plt.title("Filled Binary Mask")
plt.axis("off")
plt.show()

## Stage 9 — Distance Transform

The distance transform assigns a value to each foreground pixel based on its distance from the nearest background pixel.

The center of each solid nut produces a local maximum.  
This helps identify individual nuts even when they touch.

In [ ]:
distance_map = cv2.distanceTransform(
    filled_mask,
    cv2.DIST_L2,
    5
)

plt.figure(figsize=(5, 7))
plt.imshow(distance_map, cmap="jet")
plt.title("Distance Transform")
plt.axis("off")
plt.show()

## Stage 10 — Local Peaks

Local maxima in the distance map are used as **markers**.

Important improvement:
- Peaks are **not tracked directly**.
- They are only used as seeds for Watershed.
- The final tracked point is the centroid of each separated watershed object.

In [ ]:
PEAK_MIN_DISTANCE = 8
PEAK_THRESHOLD = 2.5

peak_coordinates = peak_local_max(
    distance_map,
    min_distance=PEAK_MIN_DISTANCE,
    threshold_abs=PEAK_THRESHOLD,
    labels=filled_mask
)

print("Detected local peaks:", len(peak_coordinates))

## Stage 11 — Watershed Separation

Watershed separates merged/touching objects using local peaks as markers.

Pipeline:

`Filled Mask → Distance Transform → Local Peaks → Markers → Watershed → Separated Nuts`

In [ ]:
peak_mask = np.zeros_like(filled_mask, dtype=bool)

if len(peak_coordinates) > 0:
    peak_mask[
        peak_coordinates[:, 0],
        peak_coordinates[:, 1]
    ] = True

markers, _ = ndi.label(peak_mask)

labels_ws = watershed(
    -distance_map,
    markers,
    mask=filled_mask.astype(bool)
)

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(filled_mask, cmap="gray")
plt.title("Before Watershed")

plt.subplot(1, 2, 2)
plt.imshow(labels_ws, cmap="nipy_spectral")
plt.title("After Watershed")

for i in [1, 2]:
    plt.subplot(1, 2, i)
    plt.axis("off")

plt.tight_layout()
plt.show()

## Stage 12 — Tracking & Counting Logic

Tracking is required because the same nut appears in many consecutive frames.

The final logic uses:
- Motion prediction
- Hungarian assignment for global matching
- Track confirmation
- One-time counting flag

### Final Counting Rule

Because the conveyor moves **upward**:

```python
if old_y > LINE_Y and new_y <= LINE_Y:
    total_count += 1
```

Each confirmed nut is counted only once.

# Final System Code

Run the following cell to execute the complete system and save an annotated output video:

**Output file:** `nut_counting_validation.mp4`

In [ ]:
# ============================================================
# FINAL NUT COUNTING SYSTEM
# UPWARD CONVEYOR MOTION + VALIDATION VIDEO
# ============================================================

VIDEO_PATH = "nuts_video.mp4"

LINE_Y = 300
PROCESSING_BOTTOM = 370

SATURATION_THRESHOLD = 100

MIN_BLOB_AREA = 60
MIN_NUT_AREA = 80
MAX_NUT_AREA = 450

PEAK_MIN_DISTANCE = 8
PEAK_THRESHOLD = 2.5

MAX_MATCH_DISTANCE = 50
MAX_MISSED_FRAMES = 15
MIN_CONFIRM_FRAMES = 3

tracks = {}
next_id = 1
total_count = 0

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError("Could not open video file.")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

writer = cv2.VideoWriter(
    "nut_counting_validation.mp4",
    fourcc,
    fps,
    (width, height)
)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    # --------------------------------------------------------
    # 1) ROI
    # --------------------------------------------------------
    roi = frame[:PROCESSING_BOTTOM, :]

    # --------------------------------------------------------
    # 2) HSV SEGMENTATION
    # --------------------------------------------------------
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    _, saturation, _ = cv2.split(hsv)

    mask = cv2.inRange(
        saturation,
        0,
        SATURATION_THRESHOLD
    )

    # --------------------------------------------------------
    # 3) MORPHOLOGICAL OPENING
    # --------------------------------------------------------
    kernel = np.ones((3, 3), np.uint8)

    clean_mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        kernel
    )

    # --------------------------------------------------------
    # 4) REMOVE SMALL BLOBS
    # --------------------------------------------------------
    contours, _ = cv2.findContours(
        clean_mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    valid_mask = np.zeros_like(clean_mask)

    for contour in contours:

        area = cv2.contourArea(contour)

        if area >= MIN_BLOB_AREA:
            cv2.drawContours(
                valid_mask,
                [contour],
                -1,
                255,
                cv2.FILLED
            )

    # --------------------------------------------------------
    # 5) FILL HOLES
    # --------------------------------------------------------
    filled_mask = valid_mask.copy()

    external_contours, _ = cv2.findContours(
        filled_mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    cv2.drawContours(
        filled_mask,
        external_contours,
        -1,
        255,
        cv2.FILLED
    )

    # --------------------------------------------------------
    # 6) DISTANCE TRANSFORM
    # --------------------------------------------------------
    distance_map = cv2.distanceTransform(
        filled_mask,
        cv2.DIST_L2,
        5
    )

    # --------------------------------------------------------
    # 7) LOCAL PEAKS
    # --------------------------------------------------------
    peak_coordinates = peak_local_max(
        distance_map,
        min_distance=PEAK_MIN_DISTANCE,
        threshold_abs=PEAK_THRESHOLD,
        labels=filled_mask
    )

    peak_mask = np.zeros_like(
        filled_mask,
        dtype=bool
    )

    if len(peak_coordinates) > 0:
        peak_mask[
            peak_coordinates[:, 0],
            peak_coordinates[:, 1]
        ] = True

    # --------------------------------------------------------
    # 8) WATERSHED
    # --------------------------------------------------------
    markers, _ = ndi.label(peak_mask)

    labels_ws = watershed(
        -distance_map,
        markers,
        mask=filled_mask.astype(bool)
    )

    # --------------------------------------------------------
    # 9) EXTRACT CENTROIDS
    # --------------------------------------------------------
    detections = []

    for label_id in range(1, labels_ws.max() + 1):

        object_mask = (
            labels_ws == label_id
        ).astype(np.uint8) * 255

        object_contours, _ = cv2.findContours(
            object_mask,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        if len(object_contours) == 0:
            continue

        c = max(
            object_contours,
            key=cv2.contourArea
        )

        area = cv2.contourArea(c)

        if area < MIN_NUT_AREA:
            continue

        if area > MAX_NUT_AREA:
            continue

        M = cv2.moments(c)

        if M["m00"] == 0:
            continue

        cx = M["m10"] / M["m00"]
        cy = M["m01"] / M["m00"]

        if cx < 5 or cx > frame.shape[1] - 6:
            continue

        detections.append(
            (float(cx), float(cy))
        )

    # --------------------------------------------------------
    # 10) TRACK PREDICTION
    # --------------------------------------------------------
    track_ids = list(tracks.keys())
    predictions = []

    for object_id in track_ids:

        data = tracks[object_id]

        x, y = data["center"]
        vx, vy = data["velocity"]

        predictions.append(
            (x + vx, y + vy)
        )

    # --------------------------------------------------------
    # 11) HUNGARIAN GLOBAL MATCHING
    # --------------------------------------------------------
    matched_track_indices = set()
    matched_detection_indices = set()

    if len(predictions) > 0 and len(detections) > 0:

        cost_matrix = np.full(
            (
                len(predictions),
                len(detections)
            ),
            9999.0,
            dtype=np.float32
        )

        for i, (px, py) in enumerate(predictions):

            for j, (dx, dy) in enumerate(detections):

                spatial_distance = np.sqrt(
                    (dx - px) ** 2
                    +
                    (dy - py) ** 2
                )

                old_y = tracks[
                    track_ids[i]
                ]["center"][1]

                direction_penalty = 0

                # Conveyor direction is UPWARD.
                if dy > old_y + 10:
                    direction_penalty = 20

                cost_matrix[i, j] = (
                    spatial_distance
                    +
                    direction_penalty
                )

        rows, cols = linear_sum_assignment(
            cost_matrix
        )

        for row, col in zip(rows, cols):

            if cost_matrix[row, col] > MAX_MATCH_DISTANCE:
                continue

            object_id = track_ids[row]

            new_x, new_y = detections[col]

            old_x, old_y = tracks[
                object_id
            ]["center"]

            measured_vx = new_x - old_x
            measured_vy = new_y - old_y

            old_vx, old_vy = tracks[
                object_id
            ]["velocity"]

            vx = (
                0.65 * old_vx
                +
                0.35 * measured_vx
            )

            vy = (
                0.65 * old_vy
                +
                0.35 * measured_vy
            )

            tracks[object_id]["center"] = (
                new_x,
                new_y
            )

            tracks[object_id]["velocity"] = (
                vx,
                vy
            )

            tracks[object_id]["missed"] = 0
            tracks[object_id]["age"] += 1

            if tracks[object_id]["age"] >= MIN_CONFIRM_FRAMES:
                tracks[object_id]["confirmed"] = True

            # ------------------------------------------------
            # FINAL COUNTING RULE
            # Upward motion:
            # BELOW line -> ABOVE line
            # ------------------------------------------------
            if (
                tracks[object_id]["confirmed"]
                and
                not tracks[object_id]["counted"]
                and
                old_y > LINE_Y
                and
                new_y <= LINE_Y
            ):

                total_count += 1

                tracks[object_id]["counted"] = True

                print(
                    f"NUT ID {object_id} COUNTED"
                    f" -> TOTAL = {total_count}"
                )

            matched_track_indices.add(row)
            matched_detection_indices.add(col)

    # --------------------------------------------------------
    # 12) HANDLE MISSED TRACKS
    # --------------------------------------------------------
    for index, object_id in enumerate(track_ids):

        if index in matched_track_indices:
            continue

        tracks[object_id]["missed"] += 1

        x, y = tracks[object_id]["center"]
        vx, vy = tracks[object_id]["velocity"]

        tracks[object_id]["center"] = (
            x + vx,
            y + vy
        )

        if tracks[object_id]["missed"] > MAX_MISSED_FRAMES:
            del tracks[object_id]

    # --------------------------------------------------------
    # 13) CREATE NEW TRACKS
    # --------------------------------------------------------
    for detection_index, detection in enumerate(detections):

        if detection_index in matched_detection_indices:
            continue

        x, y = detection

        tracks[next_id] = {
            "center": (x, y),
            "velocity": (0.0, 0.0),
            "missed": 0,
            "age": 1,
            "confirmed": False,
            "counted": False
        }

        next_id += 1

    # --------------------------------------------------------
    # 14) VISUALIZATION
    # --------------------------------------------------------
    display = frame.copy()

    cv2.line(
        display,
        (0, LINE_Y),
        (display.shape[1], LINE_Y),
        (0, 0, 255),
        2
    )

    cv2.putText(
        display,
        "COUNTING LINE",
        (10, LINE_Y - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (0, 0, 255),
        2
    )

    for object_id, data in tracks.items():

        if data["missed"] > 0:
            continue

        x, y = data["center"]
        x = int(x)
        y = int(y)

        # Yellow = tentative
        if not data["confirmed"]:
            color = (0, 255, 255)

        # Green = confirmed
        elif not data["counted"]:
            color = (0, 255, 0)

        # Magenta = counted
        else:
            color = (255, 0, 255)

        cv2.circle(
            display,
            (x, y),
            4,
            color,
            -1
        )

        if data["confirmed"]:
            cv2.putText(
                display,
                f"ID {object_id}",
                (x + 5, y - 5),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.35,
                color,
                1
            )

    cv2.putText(
        display,
        f"TOTAL NUTS: {total_count}",
        (10, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.85,
        (255, 255, 255),
        2
    )

    # Save annotated frame
    writer.write(display)

    cv2.imshow(
        "FINAL NUT COUNTING SYSTEM",
        display
    )

    if cv2.waitKey(25) & 0xFF == ord("q"):
        break


cap.release()
writer.release()
cv2.destroyAllWindows()

print()
print("====================================")
print("FINAL TOTAL NUT COUNT =", total_count)
print("====================================")
print("Validation video saved: nut_counting_validation.mp4")

# Validation

The final system count from the tested video was:

**219 nuts**

This value must be compared against a manually verified ground truth before reporting final accuracy.

Use:

\[
\text{Counting Error} = |\text{System Count} - \text{Ground Truth}|
\]

\[
\text{Accuracy} =
\left(
1 - \frac{|\text{System Count} - \text{Ground Truth}|}{\text{Ground Truth}}
\right) \times 100
\]

> Do not claim an accuracy value until the ground-truth count is verified.

# Final Engineering Observations

1. HSV saturation was effective because the metallic nuts differ strongly from the blue conveyor.
2. Morphological opening reduced isolated noise.
3. Simple contour counting was not sufficient for touching nuts.
4. Filling nut holes improved distance-transform behavior.
5. Local peaks were used as Watershed markers.
6. Watershed separated touching nuts.
7. Tracking was required to avoid counting the same nut in every frame.
8. The original counting direction assumption was wrong; validation showed the conveyor moves **upward**.
9. The final counting condition was corrected to:

```python
old_y > LINE_Y and new_y <= LINE_Y
```

10. The cumulative counter increases once per confirmed line crossing.

# Conclusion

The project demonstrates a complete classical computer vision pipeline for an industrial conveyor application:

**Video Acquisition → HSV Segmentation → Morphology → Noise Filtering → Hole Filling → Distance Transform → Local Peaks → Watershed → Centroid Extraction → Tracking → Line Crossing → Cumulative Counting**

The system also illustrates an important engineering principle: **test, observe failure, identify the cause, modify the algorithm, and validate again.**